# Crypto Research: PyTorch RPC GPU Worker

This notebook acts as the remote GPU worker for the local crypto trading pipeline. 
It uses PyTorch Distributed RPC to execute heavy tensor operations (like LOBERT or RL models) on Colab's GPU, returning the results seamlessly to your local machine.

### Instructions:
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run the cells below.
3. Copy the public address generated by `bore` and paste it into your local `.env` or configuration.

In [ ]:
!wget https://github.com/ekzhang/bore/releases/download/v0.5.1/bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
!tar -xzf bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
!chmod +x bore
!mv bore /usr/local/bin/

In [ ]:
import os
import time
import subprocess

# Start the bore TCP tunnel in the background
print("Starting bore tunnel...")
tunnel = subprocess.Popen(["bore", "local", "29500", "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# Read the assigned port from the tunnel output
assigned_port = None
for i in range(10):
    line = tunnel.stdout.readline()
    if "listening at" in line:
        assigned_port = line.strip().split(":")[-1]
        break
    time.sleep(1)

if assigned_port:
    print("\n" + "="*60)
    print(f"✅ SUCCESS! Tunnel active.")
    print(f"⚠️ PASTE THIS INTO YOUR LOCAL CONFIG: tcp://bore.pub:{assigned_port}")
    print("="*60 + "\n")
else:
    print("Failed to start bore tunnel. Please restart the cell.")

In [ ]:
import torch
import torch.distributed.rpc as rpc

os.environ['MASTER_ADDR'] = '0.0.0.0'
os.environ['MASTER_PORT'] = '29500'

print("Initializing PyTorch RPC worker...")

# The options allow the local master to connect to this worker over the public tunnel
options = rpc.TensorPipeRpcBackendOptions(
    init_method=f"tcp://0.0.0.0:29500"
)

# Initialize the worker node. It will block and wait for the master to connect.
rpc.init_rpc(
    "worker0",
    rank=1,
    world_size=2,
    rpc_backend_options=options
)

print("RPC Worker is running and ready for tensor operations!")

# Keep the worker alive
rpc.shutdown()
